Student Dropout Prediction
#
Exploratory Data Analysis, preprocessing, model training,
evaluation, and feature importance analysis.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

In [ ]:
CAT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
DATA_PATH = "../data/students.csv"
MODEL_DIR = "../model"
STATIC_DIR = "../static"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(STATIC_DIR, exist_ok=True)

## Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## Data Quality Check

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None")
print("Duplicate rows:", df.duplicated().sum())

## Target Variable Distribution

In [ ]:
counts = df["dropout"].value_counts().sort_index()
pct = counts / counts.sum() * 100

print("Class distribution:")
print(pct.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["Stayed (0)", "Dropped out (1)"],
    counts.values,
    color=["red", "green"],
    width=0.55,
)

for bar, percentage in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f"{percentage:.1f}%",
        ha="center",
        color="#0b0b0b",
        fontsize=10,
    )

ax.set_title("Target Class Balance", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Univariate Analysis — Numerical Features

In [ ]:
numeric_cols = [
    "age", "distance_from_home_km", "high_school_gpa",
    "entrance_exam_score", "prior_failures", "attendance_rate",
    "study_hours_per_week", "current_gpa",
    "lms_login_freq_per_week", "counseling_visits",
]

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(13, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=50, color="blue", edgecolor="#fcfcfb")
    axes[i].set_title(col, loc="left", fontsize=10)
    axes[i].spines[["top", "right"]].set_visible(False)

for j in range(len(numeric_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## Univariate Analysis — Categorical Features

In [ ]:
cat_cols = [
    "gender", "family_income_level", "parental_education",
    "has_scholarship", "part_time_job", "extracurricular_activities",
]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = df[col].astype(str).value_counts()
    colors = [CAT_COLORS[j % len(CAT_COLORS)] for j in range(len(vc))]
    axes[i].bar(vc.index, vc.values, color=colors, width=0.5)
    axes[i].set_title(col, loc="left", fontsize=10)
    axes[i].spines[["top", "right"]].set_visible(False)
    axes[i].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## Bivariate Analysis — Numerical Features vs Dropout

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(13, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data0 = df.loc[df["dropout"] == 0, col]
    data1 = df.loc[df["dropout"] == 1, col]

    bp = axes[i].boxplot(
        [data0, data1],
        tick_labels=["Stayed (0)", "Dropped out (1)"],
        patch_artist=True,
        widths=0.5,
    )

    for patch, color in zip(bp["boxes"], ["green", "red"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    axes[i].set_title(col, loc="left", fontsize=10)
    axes[i].spines[["top", "right"]].set_visible(False)

for j in range(len(numeric_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## Bivariate Analysis — Categorical Features vs Dropout Rate

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

overall_dropout_rate = df["dropout"].mean() * 100

for i, col in enumerate(cat_cols):
    rate = df.groupby(col)["dropout"].mean().sort_values(ascending=False) * 100

    axes[i].bar(rate.index.astype(str), rate.values, color="blue")
    axes[i].axhline(
        overall_dropout_rate,
        color="#52514e",
        linestyle="dashed",
        linewidth=1,
    )
    axes[i].set_title(f"Dropout Rate by {col}", loc="left", fontsize=10)
    axes[i].set_ylabel("Dropout Rate (%)")
    axes[i].spines[["top", "right"]].set_visible(False)
    axes[i].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
print(
    f"Dashed line = overall average dropout rate "
    f"({overall_dropout_rate:.2f}%)"
)

## Correlation Heatmap

In [ ]:
corr_cols = numeric_cols + ["dropout"]
corr = df[corr_cols].corr()
corr

In [ ]:
cmap = mcolors.LinearSegmentedColormap.from_list(
    "div", ["blue", "#f0efec", "red"]
)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        value = corr.values[i, j]
        ax.text(
            j, i, f"{value:.2f}",
            ha="center",
            fontsize=7,
            color="#0b0b0b" if abs(value) < 0.6 else "#ffffff",
        )

plt.colorbar(im, ax=ax, label="Pearson Correlation")
ax.set_title("Correlation Matrix", loc="left")
plt.tight_layout()
plt.show()

In [ ]:
print("Strongest correlations with dropout:")
print(
    corr["dropout"]
    .drop("dropout")
    .sort_values(key=abs, ascending=False)
    .round(2)
)

## Outlier Detection

In [ ]:
def iqr_outlier_count(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return ((series < lower) | (series > upper)).sum()

In [ ]:
outlier_report = pd.Series({
    col: iqr_outlier_count(df[col])
    for col in numeric_cols
}).sort_values(ascending=False)

outlier_report.to_frame("outlier_count")

## Categorical Encoding

In [ ]:
df_enc = pd.get_dummies(df, columns=cat_cols)

feature_cols = [
    c for c in df_enc.columns
    if c not in ("student_id", "dropout")
]

X = df_enc[feature_cols]
y = df_enc["dropout"]

print("Number of features:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Model Selection and Training

In [ ]:
models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
    ),
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train_s, y_train)

    preds = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]

    cv_auc = cross_val_score(
        model,
        X_train_s,
        y_train,
        cv=5,
        scoring="roc_auc",
    ).mean()

    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
        "cv_auc": cv_auc,
    }

## Model Evaluation

In [ ]:
print(json.dumps(results, indent=2))

In [ ]:
results_df = (
    pd.DataFrame(results)
    .T
    .sort_values("roc_auc", ascending=False)
)

results_df

## Select Best Model

In [ ]:
best_name = max(
    results,
    key=lambda k: results[k]["roc_auc"],
)

best_model = models[best_name]

print(f"Best model: {best_name}")

## Classification Report

In [ ]:
best_predictions = best_model.predict(X_test_s)

print(
    classification_report(
        y_test,
        best_predictions,
        zero_division=0,
    )
)

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_predictions)
cm

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Stayed (0)", "Dropped out (1)"])
ax.set_yticklabels(["Stayed (0)", "Dropped out (1)"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix", loc="left")

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)

plt.colorbar(im)
plt.tight_layout()
plt.show()

## Save Model Artifacts

In [ ]:
model_path = os.path.join(MODEL_DIR, "model.pkl")
scaler_path = os.path.join(MODEL_DIR, "scaler.pkl")
features_path = os.path.join(MODEL_DIR, "feature_cols.json")
metrics_path = os.path.join(MODEL_DIR, "metrics.json")

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)

with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

with open(metrics_path, "w") as f:
    json.dump(
        {"best_model": best_name, "results": results},
        f,
        indent=2,
    )

print("Saved:")
print(model_path)
print(scaler_path)
print(features_path)
print(metrics_path)

## Feature Importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
else:
    importances = np.abs(best_model.coef_[0])

imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances,
})

imp_df = (
    imp_df
    .sort_values("importance", ascending=False)
    .head(10)
    .sort_values("importance", ascending=True)
)

imp_df

## Plot Top 10 Dropout Prediction Features

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 5),
    facecolor="#fcfcfb",
)

ax.set_facecolor("#fcfcfb")

ax.barh(
    imp_df["feature"],
    imp_df["importance"],
    color="#2a78d6",
    height=0.6,
)

ax.set_title(
    "Top 10 Features Associated with Dropout Prediction",
    color="#0b0b0b",
    fontsize=13,
    loc="left",
)

ax.tick_params(colors="#52514e")

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_color("#d8d7d0")

ax.set_xlabel(
    "Relative Importance",
    color="#52514e",
)

plt.tight_layout()

feature_plot_path = os.path.join(
    STATIC_DIR,
    "feature_importance.png",
)

plt.savefig(
    feature_plot_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(f"Feature importance plot saved to: {feature_plot_path}")